In [1]:
# ============================================================
# FINAL DATASET: LEVIR-CC + LEVIR-CD
# ============================================================

from pathlib import Path
import json
import pandas as pd
import numpy as np
from PIL import Image
from tqdm.auto import tqdm

# ------------------------------------------------------------
# PATHS
# ------------------------------------------------------------

BASE = Path(r"C:\College\Dissertation\Data")

CC_ROOT = (
    BASE
    / "LEVIR-CC"
    / "Levir-CC-dataset-001"
)

CD_ROOT = (
    BASE
    / "LEVIR-CD"
    / "LEVIR-CD-20261002T144955Z-1-001"
    / "LEVIR-CD"
)

OUTPUT = BASE / "FINAL_LEVIR_DATASET"
OUTPUT.mkdir(exist_ok=True)

# ------------------------------------------------------------
# LOAD LEVIR-CC CAPTIONS
# ------------------------------------------------------------

with open(
    CC_ROOT / "LevirCCcaptions.json",
    "r",
    encoding="utf-8"
) as f:
    cc_json = json.load(f)

cc_records = cc_json["images"]

cc_data = {}

for record in cc_records:

    filename = record["filename"]

    cc_data[filename] = {
        "split": record["split"],
        "changeflag": record["changeflag"],
        "captions": [
            s["raw"].strip()
            for s in record["sentences"]
        ]
    }

print("CC caption records:", len(cc_data))


# ------------------------------------------------------------
# CD → CC TILE MAPPING
#
# Each 1024×1024 CD image = 16 exact 256×256 tiles.
#
# Tile order:
# 0  1  2  3
# 4  5  6  7
# 8  9 10 11
#12 13 14 15
# ------------------------------------------------------------

def get_cd_paths(split, filename):

    cd_a = CD_ROOT / split / "A" / filename
    cd_b = CD_ROOT / split / "B" / filename
    cd_mask = CD_ROOT / split / "label" / filename

    return cd_a, cd_b, cd_mask


# ------------------------------------------------------------
# BUILD CC IMAGE INDEX
# ------------------------------------------------------------

cc_images = {}

for split in ["train", "val", "test"]:

    split_dir = CC_ROOT / "images" / split / "A"

    for path in split_dir.glob("*.png"):

        if path.name in cc_data:
            cc_images[path.name] = {
                "A": path,
                "B": (
                    CC_ROOT
                    / "images"
                    / split
                    / "B"
                    / path.name
                ),
                "split": split
            }


# ------------------------------------------------------------
# MATCH CD TILES TO CC IMAGES
#
# We use the exact spatial correspondence already verified.
# ------------------------------------------------------------

final_rows = []

cd_counts = {
    "train": 445,
    "val": 64,
    "test": 128
}

for split in ["train", "val", "test"]:

    print(f"\nProcessing {split}...")

    for cd_num in tqdm(
        range(1, cd_counts[split] + 1)
    ):

        cd_filename = f"{split}_{cd_num}.png"

        cd_a_path, cd_b_path, mask_path = get_cd_paths(
            split,
            cd_filename
        )

        if not (
            cd_a_path.exists()
            and cd_b_path.exists()
            and mask_path.exists()
        ):
            continue

        # Read mask once
        mask = np.asarray(
            Image.open(mask_path).convert("L")
        )

        # Process 16 tiles
        for tile in range(16):

            row = tile // 4
            col = tile % 4

            y1 = row * 256
            y2 = y1 + 256

            x1 = col * 256
            x2 = x1 + 256

            tile_mask = mask[y1:y2, x1:x2]

            changed_pixels = int(
                np.count_nonzero(tile_mask > 0)
            )

            change_ratio = (
                changed_pixels / (256 * 256)
            )

            # CC filename corresponding to this tile
            cc_index = (
                cd_num - 1
            ) * 16 + tile + 1

            cc_filename = f"{split}_{cc_index:06d}.png"

            # Keep only samples with CC captions
            if cc_filename not in cc_data:
                continue

            if cc_filename not in cc_images:
                continue

            final_rows.append({
                "filename": cc_filename,
                "split": split,

                "cc_A": str(
                    cc_images[cc_filename]["A"]
                ),
                "cc_B": str(
                    cc_images[cc_filename]["B"]
                ),

                "cd_filename": cd_filename,

                "cd_A": str(cd_a_path),
                "cd_B": str(cd_b_path),
                "mask": str(mask_path),

                "tile": tile,
                "row": row,
                "col": col,

                "changeflag": int(
                    cc_data[cc_filename]["changeflag"]
                ),

                "changed_pixels": changed_pixels,
                "change_ratio": change_ratio,
                "change_percent": change_ratio * 100,

                "captions": json.dumps(
                    cc_data[cc_filename]["captions"],
                    ensure_ascii=False
                )
            })


# ------------------------------------------------------------
# FINAL DATAFRAME
# ------------------------------------------------------------

final_df = pd.DataFrame(final_rows)

print("\n" + "=" * 60)
print("FINAL DATASET")
print("=" * 60)

print("Total samples:", len(final_df))

print("\nSplit:")
print(final_df["split"].value_counts())

print("\nChange flag:")
print(final_df["changeflag"].value_counts())

print("\nChange percentage:")
print(
    final_df
    .groupby("changeflag")["change_percent"]
    .describe()
)


# ------------------------------------------------------------
# SAVE
# ------------------------------------------------------------

final_df.to_csv(
    OUTPUT / "final_dataset.csv",
    index=False
)

print(
    "\nSaved:",
    OUTPUT / "final_dataset.csv"
)

CC caption records: 10077

Processing train...


  0%|          | 0/445 [00:00<?, ?it/s]


Processing val...


  0%|          | 0/64 [00:00<?, ?it/s]


Processing test...


  0%|          | 0/128 [00:00<?, ?it/s]


FINAL DATASET
Total samples: 7264

Split:
split
train    6345
val       919
Name: count, dtype: int64

Change flag:
changeflag
1    3874
0    3390
Name: count, dtype: int64

Change percentage:
             count      mean       std  min       25%       50%        75%  \
changeflag                                                                   
0           3390.0  0.035288  0.336001  0.0  0.000000  0.000000   0.000000   
1           3874.0  9.491513  9.971221  0.0  1.378632  5.696106  15.517807   

                  max  
changeflag             
0           13.232422  
1           70.484924  

Saved: C:\College\Dissertation\Data\FINAL_LEVIR_DATASET\final_dataset.csv


In [2]:
# ============================================================
# FINAL DATASET VALIDATION
# ============================================================

print("Total:", len(final_df))

print("\nMissing values:")
print(
    final_df[
        ["cc_A", "cc_B", "cd_A", "cd_B", "mask", "captions"]
    ]
    .isna()
    .sum()
)

print("\nSamples per split:")
print(final_df["split"].value_counts())

print("\nChange distribution:")
print(
    final_df.groupby(
        ["split", "changeflag"]
    ).size()
)

print("\nFirst sample:")
display(final_df.iloc[0])

print("\nLast sample:")
display(final_df.iloc[-1])

Total: 7264

Missing values:
cc_A        0
cc_B        0
cd_A        0
cd_B        0
mask        0
captions    0
dtype: int64

Samples per split:
split
train    6345
val       919
Name: count, dtype: int64

Change distribution:
split  changeflag
train  0             2938
       1             3407
val    0              452
       1              467
dtype: int64

First sample:


filename                                           train_000001.png
split                                                         train
cc_A              C:\College\Dissertation\Data\LEVIR-CC\Levir-CC...
cc_B              C:\College\Dissertation\Data\LEVIR-CC\Levir-CC...
cd_filename                                             train_1.png
cd_A              C:\College\Dissertation\Data\LEVIR-CD\LEVIR-CD...
cd_B              C:\College\Dissertation\Data\LEVIR-CD\LEVIR-CD...
mask              C:\College\Dissertation\Data\LEVIR-CD\LEVIR-CD...
tile                                                              0
row                                                               0
col                                                               0
changeflag                                                        0
changed_pixels                                                    0
change_ratio                                                    0.0
change_percent                                  


Last sample:


filename                                             val_001024.png
split                                                           val
cc_A              C:\College\Dissertation\Data\LEVIR-CC\Levir-CC...
cc_B              C:\College\Dissertation\Data\LEVIR-CC\Levir-CC...
cd_filename                                              val_64.png
cd_A              C:\College\Dissertation\Data\LEVIR-CD\LEVIR-CD...
cd_B              C:\College\Dissertation\Data\LEVIR-CD\LEVIR-CD...
mask              C:\College\Dissertation\Data\LEVIR-CD\LEVIR-CD...
tile                                                             15
row                                                               3
col                                                               3
changeflag                                                        0
changed_pixels                                                    0
change_ratio                                                    0.0
change_percent                                  

In [3]:
# ============================================================
# CC CAPTION LABEL vs CD MASK AGREEMENT
# ============================================================

print("CC no-change but CD mask has change:")
print(
    ((final_df["changeflag"] == 0) &
     (final_df["changed_pixels"] > 0)).sum()
)

print("\nCC change but CD mask is empty:")
print(
    ((final_df["changeflag"] == 1) &
     (final_df["changed_pixels"] == 0)).sum()
)

print("\nPerfect agreement:")
print(
    (
        ((final_df["changeflag"] == 0) &
         (final_df["changed_pixels"] == 0))
        |
        ((final_df["changeflag"] == 1) &
         (final_df["changed_pixels"] > 0))
    ).mean()
)

CC no-change but CD mask has change:
139

CC change but CD mask is empty:
518

Perfect agreement:
0.9095539647577092
